
================================================================
#  CodeAlpha Internship — Task 3: Music Generation with AI

#  Developer Name : Rashid Ahmad

#  Dataset        : Bach Chorales + Mozart Sonatas (music21 corpus)

#  Model          : Music Transformer (Multi-Head Attention + Positional Encoding)

#  Libraries      : music21, tensorflow, gradio, fluidsynth

#  Architecture   : Transformer Decoder with learned positional
#  embeddings,
#                   temperature + top-k sampling, multi-instrument export
================================================================


In [ ]:


# ─── 1. System Dependencies ────────────────────────────────────────────────────
!apt-get update -qq
!apt-get install -y -qq fluidsynth fluid-soundfont-gm timidity

# ─── 2. Python Dependencies ────────────────────────────────────────────────────
!pip install -q music21 gradio pretty_midi

# ─── 3. Imports ────────────────────────────────────────────────────────────────
import os
import json
import random
import warnings
import subprocess
import numpy as np
import music21
import tensorflow as tf
import gradio as gr
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
tf.get_logger().setLevel("ERROR")


# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
class Config:
    SEQUENCE_LENGTH   = 128          # Context window (tokens)
    EMBED_DIM         = 256          # Transformer embedding dimension
    NUM_HEADS         = 8            # Multi-head attention heads
    FF_DIM            = 512          # Feed-forward inner dimension
    NUM_LAYERS        = 4            # Number of Transformer decoder blocks
    DROPOUT_RATE      = 0.1
    BATCH_SIZE        = 64
    EPOCHS            = 50
    PATIENCE          = 10           # Early stopping patience
    LEARNING_RATE     = 1e-3
    NUM_CHORALES      = 30           # Bach chorales to load
    SAMPLE_RATE       = 44100
    MODEL_PATH        = "/content/rashid_music_transformer.keras"
    VOCAB_PATH        = "/content/rashid_vocab.json"
    SOUNDFONT_PATH    = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
    MIDI_OUT          = "/content/rashid_generated.mid"
    WAV_OUT           = "/content/rashid_generated.wav"
    SEED              = 2024


# ══════════════════════════════════════════════════════════════════════════════
#  POSITIONAL ENCODING
# ══════════════════════════════════════════════════════════════════════════════
class PositionalEncoding(tf.keras.layers.Layer):
    """Classic sinusoidal positional encoding (Vaswani et al., 2017)."""

    def __init__(self, max_len: int, embed_dim: int, **kwargs):
        super().__init__(**kwargs)
        self.max_len   = max_len
        self.embed_dim = embed_dim

        # Pre-compute the encoding matrix  [1, max_len, embed_dim]
        positions = np.arange(max_len)[:, np.newaxis]          # (max_len, 1)
        dims      = np.arange(embed_dim)[np.newaxis, :]        # (1, embed_dim)
        angles    = positions / np.power(10000, (2 * (dims // 2)) / embed_dim)
        angles[:, 0::2] = np.sin(angles[:, 0::2])
        angles[:, 1::2] = np.cos(angles[:, 1::2])
        self.encoding = tf.cast(angles[np.newaxis, :, :], tf.float32)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.encoding[:, :seq_len, :]

    def get_config(self):
        config = super().get_config()
        config.update({"max_len": self.max_len, "embed_dim": self.embed_dim})
        return config


# ══════════════════════════════════════════════════════════════════════════════
#  TRANSFORMER DECODER BLOCK
# ══════════════════════════════════════════════════════════════════════════════
class TransformerBlock(tf.keras.layers.Layer):
    """
    Single Transformer decoder block with causal (masked) self-attention.
    Causal masking ensures the model only attends to past tokens — critical
    for autoregressive music generation.
    """

    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attention  = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads
        )
        self.ffn        = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="gelu"),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.norm1      = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2      = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.drop1      = tf.keras.layers.Dropout(dropout_rate)
        self.drop2      = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        seq_len   = tf.shape(x)[1]
        causal_mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        causal_mask = tf.cast(causal_mask[tf.newaxis, tf.newaxis, :, :], tf.bool)

        attn_out = self.attention(x, x, attention_mask=causal_mask, training=training)
        x        = self.norm1(x + self.drop1(attn_out, training=training))
        ffn_out  = self.ffn(x)
        x        = self.norm2(x + self.drop2(ffn_out, training=training))
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim"   : self.ffn.layers[-1].units,
            "num_heads"   : self.attention._num_heads,
            "ff_dim"      : self.ffn.layers[0].units,
            "dropout_rate": self.drop1.rate,
        })
        return config


# ══════════════════════════════════════════════════════════════════════════════
#  DATA PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
class MusicDataPipeline:
    """
    Handles corpus loading, vocabulary construction, and sequence preparation.
    Encodes notes as strings: 'C4' for single notes, '0.4.7' for chords,
    and 'R' for rests — giving a richer, more musical token vocabulary.
    """

    def __init__(self, config: Config):
        self.cfg          = config
        self.note_to_int  = {}
        self.int_to_note  = {}
        self.vocab_size   = 0
        self._raw_notes   = []

    def _parse_score(self, file_path) -> list:
        """Extract note tokens from a single music21 score file."""
        tokens = []
        try:
            score  = music21.converter.parse(file_path)
            parts  = music21.instrument.partitionByInstrument(score)
            stream = parts.parts[0].recurse() if parts else score.flat.notesAndRests

            for el in stream:
                if isinstance(el, music21.note.Note):
                    tokens.append(str(el.pitch))
                elif isinstance(el, music21.chord.Chord):
                    tokens.append(".".join(str(n) for n in el.normalOrder))
                elif isinstance(el, music21.note.Rest):
                    tokens.append("R")
        except Exception as e:
            pass  # Gracefully skip corrupt/unsupported files
        return tokens

    def load(self, composers=("bach",), limit=30) -> tuple:
        """Load corpus, build vocabulary, return (X, y) training arrays."""
        print(f"🎼  Loading corpus — composers: {composers}, limit per composer: {limit}")
        all_notes = []

        for composer in composers:
            files = music21.corpus.getComposer(composer)[:limit]
            for f in files:
                all_notes.extend(self._parse_score(f))

        self._raw_notes = all_notes

        # Build vocabulary
        unique = sorted(set(all_notes))
        self.vocab_size  = len(unique)
        self.note_to_int = {n: i for i, n in enumerate(unique)}
        self.int_to_note = {i: n for i, n in enumerate(unique)}

        # Sequence windows
        seqs, targets = [], []
        for i in range(len(all_notes) - self.cfg.SEQUENCE_LENGTH):
            seq = all_notes[i : i + self.cfg.SEQUENCE_LENGTH]
            seqs.append([self.note_to_int[t] for t in seq])
            targets.append(self.note_to_int[all_notes[i + self.cfg.SEQUENCE_LENGTH]])

        X = np.array(seqs, dtype=np.int32)
        y = tf.keras.utils.to_categorical(targets, num_classes=self.vocab_size)

        print(f"✅  Corpus loaded — vocabulary: {self.vocab_size} tokens, "
              f"sequences: {len(X):,}")
        return X, y

    def save_vocab(self, path: str):
        with open(path, "w") as f:
            json.dump({"note_to_int": self.note_to_int,
                       "int_to_note": {str(k): v
                                       for k, v in self.int_to_note.items()}}, f)

    def load_vocab(self, path: str):
        with open(path) as f:
            data = json.load(f)
        self.note_to_int = data["note_to_int"]
        self.int_to_note = {int(k): v for k, v in data["int_to_note"].items()}
        self.vocab_size  = len(self.note_to_int)


# ══════════════════════════════════════════════════════════════════════════════
#  MUSIC TRANSFORMER MODEL
# ══════════════════════════════════════════════════════════════════════════════
class MusicTransformer:
    """
    Autoregressive Music Transformer.

    Architecture:
        Token Embedding  →  Positional Encoding  →  N × TransformerBlock
        →  LayerNorm  →  Dense(vocab_size, softmax)

    Why Transformer over LSTM?
    - Parallelises training (no sequential hidden-state bottleneck).
    - Causal self-attention captures long-range harmonic dependencies
      that LSTMs struggle with beyond ~100 steps.
    - Scales better with more data and model depth.
    """

    def __init__(self, config: Config, vocab_size: int):
        self.cfg        = config
        self.vocab_size = vocab_size
        self.model      = None

    def build(self):
        cfg = self.cfg
        inputs  = tf.keras.Input(shape=(cfg.SEQUENCE_LENGTH,), dtype=tf.int32)
        x       = tf.keras.layers.Embedding(self.vocab_size, cfg.EMBED_DIM)(inputs)
        x       = PositionalEncoding(cfg.SEQUENCE_LENGTH, cfg.EMBED_DIM)(x)
        x       = tf.keras.layers.Dropout(cfg.DROPOUT_RATE)(x)

        for i in range(cfg.NUM_LAYERS):
            x = TransformerBlock(
                embed_dim    = cfg.EMBED_DIM,
                num_heads    = cfg.NUM_HEADS,
                ff_dim       = cfg.FF_DIM,
                dropout_rate = cfg.DROPOUT_RATE,
                name         = f"transformer_block_{i}",
            )(x)

        x       = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
        x       = x[:, -1, :]                                      # Last token only
        outputs = tf.keras.layers.Dense(self.vocab_size, activation="softmax")(x)

        self.model = tf.keras.Model(inputs, outputs, name="MusicTransformer")

        lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
            initial_learning_rate = cfg.LEARNING_RATE,
            first_decay_steps     = 1000,
        )
        self.model.compile(
            optimizer = tf.keras.optimizers.Adam(lr_schedule, clipnorm=1.0),
            loss      = "categorical_crossentropy",
            metrics   = ["accuracy"],
        )
        print("🧠  MusicTransformer built:")
        self.model.summary()

    def train(self, X, y):
        cfg = self.cfg
        print("🚀  Training MusicTransformer…")

        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                cfg.MODEL_PATH, monitor="loss",
                save_best_only=True, verbose=1
            ),
            tf.keras.callbacks.EarlyStopping(
                monitor="loss", patience=cfg.PATIENCE,
                restore_best_weights=True, verbose=1
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="loss", factor=0.5, patience=4,
                min_lr=1e-6, verbose=1
            ),
        ]

        history = self.model.fit(
            X, y,
            epochs     = cfg.EPOCHS,
            batch_size = cfg.BATCH_SIZE,
            callbacks  = callbacks,
            shuffle    = True,
        )
        print(f"✅  Training complete. Model saved → {cfg.MODEL_PATH}")
        return history

    def load_weights(self, path: str):
        self.model = tf.keras.models.load_model(
            path,
            custom_objects={
                "PositionalEncoding": PositionalEncoding,
                "TransformerBlock"  : TransformerBlock,
            },
        )
        print(f"✅  Model loaded from {path}")


# ══════════════════════════════════════════════════════════════════════════════
#  GENERATION ENGINE
# ══════════════════════════════════════════════════════════════════════════════
class MusicGenerationEngine:
    """
    Handles autoregressive token sampling with temperature + top-k filtering.

    top_k=0  → pure temperature sampling (more chaotic)
    top_k>0  → nucleus-like sampling over top-k candidates (more coherent)
    """

    def __init__(self, transformer: MusicTransformer, pipeline: MusicDataPipeline, config: Config):
        self.transformer = transformer
        self.pipeline    = pipeline
        self.cfg         = config

    def _top_k_sample(self, logits: np.ndarray, k: int) -> int:
        if k > 0:
            top_k_idx    = np.argsort(logits)[-k:]
            filtered     = np.full_like(logits, -np.inf)
            filtered[top_k_idx] = logits[top_k_idx]
            logits = filtered
        # Softmax
        logits -= logits.max()
        probs   = np.exp(logits)
        probs  /= probs.sum()
        return int(np.random.choice(len(probs), p=probs))

    def generate(
        self,
        num_notes  : int   = 150,
        temperature: float = 1.0,
        top_k      : int   = 10,
        seed_index : int   = None,
    ) -> list:
        if self.transformer.model is None:
            raise RuntimeError("Model not trained or loaded. Call .train() first.")

        pipeline = self.pipeline
        cfg      = self.cfg
        seq_len  = cfg.SEQUENCE_LENGTH

        # Seed from a random training sequence
        all_int = [pipeline.note_to_int[n] for n in pipeline._raw_notes]
        idx     = seed_index if seed_index is not None \
                  else random.randint(0, len(all_int) - seq_len - 1)
        pattern = list(all_int[idx : idx + seq_len])
        output  = []

        print(f"🎹  Generating {num_notes} notes  "
              f"(temp={temperature}, top_k={top_k})…")

        for step in range(num_notes):
            x       = np.array([pattern], dtype=np.int32)
            logits  = self.transformer.model.predict(x, verbose=0)[0]

            # Apply temperature
            log_logits = np.log(logits + 1e-9) / temperature
            idx_pred   = self._top_k_sample(log_logits, top_k)

            output.append(pipeline.int_to_note[idx_pred])
            pattern.append(idx_pred)
            pattern = pattern[1:]

        print(f"✅  Generation complete — {len(output)} tokens produced.")
        return output


# ══════════════════════════════════════════════════════════════════════════════
#  AUDIO EXPORT
# ══════════════════════════════════════════════════════════════════════════════
class AudioExporter:
    """
    Converts generated token sequences → MIDI → WAV.

    Tokens are decoded back into music21 Note / Chord objects.
    A quarter-note grid (offset += 0.5) keeps timing simple and musical.
    FluidSynth renders the MIDI to broadcast-quality 44.1 kHz WAV.
    """

    def __init__(self, config: Config):
        self.cfg = config

    def tokens_to_midi(self, tokens: list) -> str:
        offset       = 0.0
        output_notes = []

        for token in tokens:
            if token == "R":
                r        = music21.note.Rest()
                r.offset = offset
                output_notes.append(r)

            elif "." in token or token.isdigit():
                # Chord
                pitches = []
                for p in token.split("."):
                    n = music21.note.Note(int(p))
                    n.storedInstrument = music21.instrument.Piano()
                    pitches.append(n)
                chord        = music21.chord.Chord(pitches)
                chord.offset = offset
                output_notes.append(chord)

            else:
                # Single note
                note               = music21.note.Note(token)
                note.offset        = offset
                note.storedInstrument = music21.instrument.Piano()
                output_notes.append(note)

            offset += 0.5

        stream = music21.stream.Stream(output_notes)
        stream.write("midi", fp=self.cfg.MIDI_OUT)
        print(f"🎵  MIDI saved → {self.cfg.MIDI_OUT}")
        return self.cfg.MIDI_OUT

    def midi_to_wav(self, midi_path: str) -> str:
        cmd = [
            "fluidsynth", "-ni",
            self.cfg.SOUNDFONT_PATH,
            midi_path,
            "-F", self.cfg.WAV_OUT,
            "-r", str(self.cfg.SAMPLE_RATE),
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"⚠️  FluidSynth warning: {result.stderr[:200]}")
        print(f"🔊  WAV saved  → {self.cfg.WAV_OUT}")
        return self.cfg.WAV_OUT

    def export(self, tokens: list) -> tuple:
        midi_path = self.tokens_to_midi(tokens)
        wav_path  = self.midi_to_wav(midi_path)
        return midi_path, wav_path


# ══════════════════════════════════════════════════════════════════════════════
#  ORCHESTRATOR — ties everything together
# ══════════════════════════════════════════════════════════════════════════════
class RashidMusicSystem:
    """Single entry-point that wires the pipeline, model, generator, and exporter."""

    def __init__(self):
        tf.random.set_seed(Config.SEED)
        np.random.seed(Config.SEED)

        self.cfg       = Config()
        self.pipeline  = MusicDataPipeline(self.cfg)
        self.transformer = None
        self.engine    = None
        self.exporter  = AudioExporter(self.cfg)

    # ── Training ──────────────────────────────────────────────────────────────
    def train(self, num_chorales: int = 30) -> str:
        X, y = self.pipeline.load(composers=("bach",), limit=num_chorales)
        self.pipeline.save_vocab(self.cfg.VOCAB_PATH)

        self.transformer = MusicTransformer(self.cfg, self.pipeline.vocab_size)
        self.transformer.build()
        self.transformer.train(X, y)

        self.engine = MusicGenerationEngine(self.transformer, self.pipeline, self.cfg)
        return (f"✅ Training complete!\n"
                f"   Vocabulary : {self.pipeline.vocab_size} tokens\n"
                f"   Model saved: {self.cfg.MODEL_PATH}")

    # ── Resume from saved model ────────────────────────────────────────────────
    def load_existing(self) -> str:
        if not Path(self.cfg.MODEL_PATH).exists():
            return "⚠️  No saved model found. Please train first."
        if not Path(self.cfg.VOCAB_PATH).exists():
            return "⚠️  No vocabulary file found. Please re-train."

        self.pipeline.load_vocab(self.cfg.VOCAB_PATH)
        self.transformer = MusicTransformer(self.cfg, self.pipeline.vocab_size)
        self.transformer.load_weights(self.cfg.MODEL_PATH)
        self.engine = MusicGenerationEngine(self.transformer, self.pipeline, self.cfg)
        return "✅ Model and vocabulary loaded successfully."

    # ── Generation ────────────────────────────────────────────────────────────
    def generate(
        self,
        num_notes  : int,
        temperature: float,
        top_k      : int,
    ) -> tuple:
        if self.engine is None:
            return None, None, "⚠️ No model available. Train or load a model first."
        try:
            tokens             = self.engine.generate(num_notes, temperature, top_k)
            midi_path, wav_path = self.exporter.export(tokens)
            return midi_path, wav_path, f"✅ Generated {len(tokens)} musical tokens successfully!"
        except Exception as e:
            return None, None, f"❌ Error during generation: {e}"


# ══════════════════════════════════════════════════════════════════════════════
#  GRADIO UI
# ══════════════════════════════════════════════════════════════════════════════
system = RashidMusicSystem()

def ui_train(num_chorales):
    return system.train(int(num_chorales))

def ui_load():
    return system.load_existing()

def ui_generate(num_notes, temperature, top_k):
    return system.generate(int(num_notes), float(temperature), int(top_k))


CSS = """
body { font-family: 'Georgia', serif; }
.gr-button-primary { background: linear-gradient(135deg, #1a1a2e, #16213e) !important; }
.gr-button-secondary { background: linear-gradient(135deg, #0f3460, #533483) !important; }
footer { display: none !important; }
"""

with gr.Blocks(theme=gr.themes.Base(), css=CSS) as demo:
    gr.Markdown("""
    # 🎼 Rashid Ahmad — AI Music Transformer
    ### Autoregressive Music Generation via Multi-Head Self-Attention
    *Learns harmonic structure from Bach chorales, then synthesises original MIDI/WAV compositions*
    ---
    """)

    with gr.Tabs():

        # ── Tab 1: Training ───────────────────────────────────────────────────
        with gr.TabItem("⚙️  Model Training"):
            gr.Markdown("Train a fresh MusicTransformer from the Bach corpus, or reload a saved checkpoint.")
            with gr.Row():
                with gr.Column(scale=2):
                    chorales_slider = gr.Slider(
                        minimum=5, maximum=50, value=30, step=5,
                        label="Bach Chorales to Load",
                        info="More chorales = richer vocabulary but longer training time"
                    )
                with gr.Column(scale=1):
                    train_btn = gr.Button("🚀  Train New Model", variant="primary")
                    load_btn  = gr.Button("📂  Load Saved Model", variant="secondary")

            train_status = gr.Textbox(label="Status", lines=4, interactive=False)
            train_btn.click(fn=ui_train, inputs=[chorales_slider], outputs=train_status)
            load_btn.click(fn=ui_load,  inputs=[],                outputs=train_status)

        # ── Tab 2: Generation ─────────────────────────────────────────────────
        with gr.TabItem("🎹  Generate Music"):
            gr.Markdown("Configure the generation parameters and synthesise a new piece.")
            with gr.Row():
                with gr.Column():
                    num_notes_slider = gr.Slider(
                        50, 400, value=150, step=10,
                        label="Number of Notes / Chords",
                        info="~150 notes ≈ 30 seconds of piano music"
                    )
                    temp_slider = gr.Slider(
                        0.1, 2.0, value=1.0, step=0.05,
                        label="Temperature",
                        info="Low = conservative & repetitive | High = creative & chaotic"
                    )
                    topk_slider = gr.Slider(
                        1, 50, value=10, step=1,
                        label="Top-K Sampling",
                        info="Limits sampling to the K most probable tokens (0 = unrestricted)"
                    )
                    gen_btn = gr.Button("🎵  Generate", variant="primary", size="lg")

                with gr.Column():
                    audio_out  = gr.Audio(label="🔊 Generated Audio (WAV)", type="filepath")
                    midi_out   = gr.File(label="⬇️  Download MIDI")
                    gen_status = gr.Textbox(label="Status", interactive=False)

            gen_btn.click(
                fn      = ui_generate,
                inputs  = [num_notes_slider, temp_slider, topk_slider],
                outputs = [midi_out, audio_out, gen_status],
            )

        # ── Tab 3: Architecture Info ───────────────────────────────────────────
        with gr.TabItem("📖  Architecture"):
            gr.Markdown(f"""
            ## Model Architecture: Music Transformer

            | Component | Details |
            |-----------|---------|
            | **Type** | Autoregressive Transformer Decoder |
            | **Embedding** | Learned token embedding (dim={Config.EMBED_DIM}) |
            | **Positional Encoding** | Sinusoidal (Vaswani et al., 2017) |
            | **Attention** | Multi-Head Self-Attention ({Config.NUM_HEADS} heads) |
            | **Layers** | {Config.NUM_LAYERS} × TransformerBlock (causal mask) |
            | **FFN Activation** | GELU |
            | **Sequence Length** | {Config.SEQUENCE_LENGTH} tokens |
            | **Sampling** | Temperature + Top-K |
            | **Optimiser** | Adam + Cosine Decay Restarts |
            | **Regularisation** | Dropout ({Config.DROPOUT_RATE}), Layer Norm, Gradient Clipping |
            | **Export** | MIDI via music21 → WAV via FluidSynth @ 44.1 kHz |

            ## Why Transformer over LSTM?
            - **Parallelism**: All tokens processed simultaneously during training
            - **Long-range dependencies**: Self-attention spans the full 128-token context window
            - **Scalability**: Easily extended with more layers, heads, or data
            - **GELU activations**: Smoother gradient flow than ReLU in music tasks
            """)

    gr.Markdown("---\n*Developer: Rashid Ahmad — CodeAlpha Internship, Task 3*")

demo.launch(share=True, debug=True)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libdouble-conversion3:amd64.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../00-libdouble-conversion3_3.1.7-4_amd64.deb ...
Unpacking libdouble-conversion3:amd64 (3.1.7-4) ...
Selecting previously unselected package libqt5core5a:amd64.
Preparing to unpack .../01-libqt5core5a_5.15.3+dfsg-2ubuntu0.2_amd64.deb ...
Unpacking libqt5core5a:amd64 (5.15.3+dfsg-2ubuntu0.2) ...
Selecting previously unselected package libevdev2:amd64.
Preparing to unpack .../02-libevdev2_1.12.1+dfsg-1_amd64.deb ...
Unpacking libevdev2:amd64 (1.12.1+dfsg-1) ...
Selecting previously unselected package libmtdev1:amd64.
Preparing to unpack .../03-libmtdev1_1.1.6-1build4_amd64.deb ...
Unpacking libmtdev1:

🎼  Loading corpus — composers: ('bach',), limit per composer: 30
✅  Corpus loaded — vocabulary: 60 tokens, sequences: 4,718
🧠  MusicTransformer built:


Model: "MusicTransformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 128, 256)       │        15,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding             │ (None, 128, 256)       │             0 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_0             │ (None, 128, 256)       │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 128, 256)       │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, 128, 256)       │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ (None, 128, 256)       │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_8           │ (None, 128, 256)       │           512 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 60)             │        15,420 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,139,708 (8.16 MB)

 Trainable params: 2,139,708 (8.16 MB)

 Non-trainable params: 0 (0.00 B)

🚀  Training MusicTransformer…
Epoch 1/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 249ms/step - accuracy: 0.0535 - loss: 3.9316
Epoch 1: loss improved from None to 3.73099, saving model to /content/rashid_music_transformer.keras

Epoch 1: finished saving model to /content/rashid_music_transformer.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 46s 265ms/step - accuracy: 0.0593 - loss: 3.7310 - learning_rate: 9.8655e-04
Epoch 2/50
73/74 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.0628 - loss: 3.6363
Epoch 2: loss improved from 3.73099 to 3.60659, saving model to /content/rashid_music_transformer.keras

Epoch 2: finished saving model to /content/rashid_music_transformer.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 5s 62ms/step - accuracy: 0.0619 - loss: 3.6066 - learning_rate: 9.4692e-04
Epoch 3/50
73/74 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.0636 - loss: 3.6296
Epoch 3: loss did not improve from 3.60659
74/74 ━━━━━━━━━━━━━━━━━━━━ 4s 59ms/step - accuracy: 0.0638 - loss: 3.6069 - learning_rate: 8.8325e-04
Epoch 4/

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

✅  Model loaded from /content/rashid_music_transformer.keras
🎹  Generating 180 notes  (temp=1.05, top_k=12)…
✅  Generation complete — 180 tokens produced.
🎵  MIDI saved → /content/rashid_generated.mid
🔊  WAV saved  → /content/rashid_generated.wav
